In [7]:
!pip install rasterio
!pip install shapely
!pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 200.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 193.5 MB/s eta 0:00:00


In [16]:
import math
import rasterio
import os
import shutil
from random import randint
from random import choice
from numpy import pad
import glob
import rasterio
from shapely.geometry import box
import geopandas as gpd
import numpy as np
from rasterio.windows import Window

os.mkdir("/home/ec2-user/SageMaker/data/test/")
os.mkdir("/home/ec2-user/SageMaker/data/train/")
os.mkdir("/home/ec2-user/SageMaker/data/test/images")
os.mkdir("/home/ec2-user/SageMaker/data/test/masks")
os.mkdir("/home/ec2-user/SageMaker/data/train/images")
os.mkdir("/home/ec2-user/SageMaker/data/train/masks")
os.mkdir("/home/ec2-user/SageMaker/data/train/hillshade")
os.mkdir("/home/ec2-user/SageMaker/data/train/dem")
os.mkdir("/home/ec2-user/SageMaker/data/train/smashedData")
os.mkdir("/home/ec2-user/SageMaker/data/test/hillshade")
os.mkdir("/home/ec2-user/SageMaker/data/test/dem")
os.mkdir("/home/ec2-user/SageMaker/data/test/smashedData")

FileExistsError: [Errno 17] File exists: '/home/ec2-user/SageMaker/data/test/'

In [1]:
patch_size = 512

naip_path = "/home/ec2-user/SageMaker/data/raw/South_Clear_Creek/NAIP/South_Clear_Creek_2023_NAIP_1m.tif"
naip_data = rasterio.open(naip_path)
naip = naip_data.read()

mask_path = "/home/ec2-user/SageMaker/data/raw/South_Clear_Creek/Roads_Boundary/South_Clear_Creek_Roads_Mask.tif"
mask_data = rasterio.open(mask_path)
mask = mask_data.read()

hillshade_path = "/home/ec2-user/SageMaker/data/raw/South_Clear_Creek/Lidar_DEM_Hillshade/South_Clear_Creek_BareEarth_Hillshade_1m.tif"
hillshade_data = rasterio.open(hillshade_path)
hillshade = hillshade_data.read()
dem_path = "/home/ec2-user/SageMaker/data/raw/South_Clear_Creek/Lidar_DEM_Hillshade/NormalizedElevation.tiff"
dem_data = rasterio.open(dem_path)
dem = dem_data.read()

naip_dimensions, naip_width, naip_height = naip.shape
naip_rows = math.floor(naip_width / patch_size)
naip_cols = math.floor(naip_height / patch_size)

mask_dimensions, mask_width, mask_height = mask.shape
mask_rows = math.floor(mask_width / patch_size)
mask_cols = math.floor(mask_height / patch_size)

hillshade_dimensions, hillshade_width, hillshade_height = hillshade.shape
hillshade_rows = math.floor(hillshade_width / patch_size)
hillshade_cols = math.floor(hillshade_height / patch_size)

dem_dimensions, dem_width, dem_height = dem.shape
dem_rows = math.floor(dem_width / patch_size)
dem_cols = math.floor(dem_height / patch_size)

for row in range(naip_rows):
    y_offset = (row * patch_size)
    for col in range(naip_cols):
        x_offset = (col * patch_size)
        window = Window(x_offset, y_offset, patch_size, patch_size) 
        patch = naip_data.read(window=window)

        patch_meta = naip_data.meta.copy()
        patch_meta.update({
            "height": patch_size,
            "width": patch_size,
            "transform": rasterio.windows.transform(window, naip_data.transform)
        })

        patch[patch == 1.79e+308] = 0  # Set null values to zero

        patch_path = "/home/ec2-user/SageMaker/data/train/images/SCC_NAIP_1m_patch" + str(row) + "-" + str(col) + ".tif"
        with rasterio.open(patch_path, "w", **patch_meta) as writer:
            writer.write(patch)

for row in range(mask_rows):
    y_offset = row * patch_size
    for col in range(mask_cols):
        x_offset = col * patch_size
        
        window = Window(x_offset, y_offset, patch_size, patch_size) 
        patch = mask_data.read(window=window)

        patch_meta = mask_data.meta.copy()
        patch_meta.update({
            "height": patch_size,
            "width": patch_size,
            "transform": rasterio.windows.transform(window, mask_data.transform)
        })

        patch[patch == 1.79e+308] = 0  # Set null values to zero

        patch_path = "/home/ec2-user/SageMaker/data/train/masks/SCC_mask_patch" + str(row) + "-" + str(col) + ".tif"
        with rasterio.open(patch_path, "w", **patch_meta) as writer:
            writer.write(patch)

for row in range(hillshade_rows):
    y_offset = row * patch_size
    for col in range(hillshade_cols):
        x_offset = col * patch_size
        
        window = Window(x_offset, y_offset, patch_size, patch_size) 
        patch = hillshade_data.read(window=window)

        patch_meta = hillshade_data.meta.copy()
        patch_meta.update({
            "height": patch_size,
            "width": patch_size,
            "transform": rasterio.windows.transform(window, hillshade_data.transform)
        })

        patch[patch == 1.79e+308] = 0  # Set null values to zero

        patch_path = "/home/ec2-user/SageMaker/data/train/hillshade/SCC_hillshade_patch" + str(row) + "-" + str(col) + ".tif"
        with rasterio.open(patch_path, "w", **patch_meta) as writer:
            writer.write(patch)

for row in range(dem_rows):
    y_offset = row * patch_size
    for col in range(dem_cols):
        x_offset = col * patch_size
        
        window = Window(x_offset, y_offset, patch_size, patch_size) 
        patch = dem_data.read(window=window)

        patch_meta = dem_data.meta.copy()
        patch_meta.update({
            "height": patch_size,
            "width": patch_size,
            "transform": rasterio.windows.transform(window, dem_data.transform)
        })

        patch[patch == 1.79e+308] = 0  # Set null values to zero

        patch_path = "/home/ec2-user/SageMaker/data/train/dem/SCC_dem_patch" + str(row) + "-" + str(col) + ".tif"
        with rasterio.open(patch_path, "w", **patch_meta) as writer:
            writer.write(patch)

# Set the folder containing GeoTIFF files and the shapefile path.
geotiff_folder = "/home/ec2-user/SageMaker/data/train/images"
shapefile_path = "/home/ec2-user/SageMaker/data/raw/South_Clear_Creek/Roads_Boundary/South_Clear_Creek_Bounds.shp"
# Load the shapefile using GeoPandas
shp = gpd.read_file(shapefile_path)

# Loop through each GeoTIFF file
for row in range(naip_rows):
    for col in range(naip_cols):
        patch_id = str(row) + "-" + str(col)
        img = "/home/ec2-user/SageMaker/data/train/images/SCC_NAIP_1m_patch" + patch_id + ".tif"
        mask = "/home/ec2-user/SageMaker/data/train/masks/SCC_mask_patch" + patch_id + ".tif"
        hillshade = "/home/ec2-user/SageMaker/data/train/hillshade/SCC_hillshade_patch" + patch_id + ".tif"
        dem = "/home/ec2-user/SageMaker/data/train/dem/SCC_dem_patch" + patch_id + ".tif"
        
        with rasterio.open(img) as src:
            # Get the raster's coordinate reference system (CRS)
            raster_crs = src.crs
            
            # Reproject the shapefile if the CRSs don't match
            if shp.crs != raster_crs:
                shp_reproj = shp.to_crs(raster_crs)
                shp_union = shp_reproj.unary_union
            else:
                shp_union = shp.unary_union

            # Get the raster's bounding box and convert it to a shapely geometry
            bounds = src.bounds
            raster_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
            
            # Check intersection between the raster bounding box and the shapefile geometry
            intersects = raster_bbox.intersects(shp_union)
            if not (intersects):
                os.remove(img)
                os.remove(mask)
                os.remove(hillshade)
                os.remove(dem)

picked_patches = set()
twenty_percent = int(0.2 * len(os.listdir("/home/ec2-user/SageMaker/data/train/images")))

for i in range(twenty_percent):
    file_list = os.listdir("/home/ec2-user/SageMaker/data/train/images")

    filename = choice(file_list)
    file_extension = len(filename) - 4
    patch_id = ""
    for i, char in enumerate(filename):
        if char == "p": # I'm cutting off at the p don't question it
            patch_id = filename[i:file_extension]
            break

    while(patch_id in picked_patches):
        filename = choice(file_list)
        file_extension = len(filename) - 4
        patch_id = ""
        for i, char in enumerate(filename):
            if char.isdigit():
                patch_id = filename[i:file_extension]
                break

    picked_patches.add(patch_id)

    source_img = "/home/ec2-user/SageMaker/data/train/images/SCC_NAIP_1m_" + patch_id + ".tif"
    source_mask = "/home/ec2-user/SageMaker/data/train/masks/SCC_mask_" + patch_id + ".tif"
    source_hillshade = "/home/ec2-user/SageMaker/data/train/hillshade/SCC_hillshade_" + patch_id + ".tif"
    source_dem = "/home/ec2-user/SageMaker/data/train/dem/SCC_dem_" + patch_id + ".tif"
    dest_img = "/home/ec2-user/SageMaker/data/test/images/SCC_NAIP_1m_" + patch_id + ".tif"
    dest_mask = "/home/ec2-user/SageMaker/data/test/masks/SCC_mask_" + patch_id + ".tif"
    dest_hillshade = "/home/ec2-user/SageMaker/data/test/hillshade/SCC_hillshade_" + patch_id + ".tif"
    dest_dem = "/home/ec2-user/SageMaker/data/test/dem/SCC_dem_" + patch_id + ".tif"

    shutil.copy2(source_img, dest_img)
    shutil.copy2(source_mask, dest_mask)
    shutil.copy2(source_hillshade, dest_hillshade)
    shutil.copy2(source_dem, dest_dem)

    os.remove(source_img)
    os.remove(source_mask)
    os.remove(source_hillshade)
    os.remove(source_dem)


NameError: name 'rasterio' is not defined